# 05 fastai

## Student-authored explanation

Write the method and result interpretation here in your own words.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
if not (REPO / "src" / "data.py").exists():
    REPO = Path("/content/credit-card-fraud-detection")
    if not (REPO / "src" / "data.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git",
             str(REPO)],
            check=True,
        )
os.chdir(REPO)
%pip install -q -r requirements/base.txt -r requirements/fastai.txt
!python src/data.py
sys.path.insert(0, str(REPO / "src"))


In [1]:
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from fastai.tabular.all import (
    CategoryBlock,
    CrossEntropyLossFlat,
    EarlyStoppingCallback,
    TabularPandas,
    tabular_config,
    tabular_learner,
    valley,
)

from data import FEATURES, ROOT, load_splits
from evaluate import evaluate_model, permutation_importance

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

(X_train, y_train, X_val, y_val, X_test, y_test,
 row_id_val, row_id_test, amount_val, amount_test) = load_splits()

train_df = X_train.reset_index(drop=True).copy()
train_df["Class"] = np.asarray(y_train, dtype=np.int64)
valid_df = X_val.reset_index(drop=True).copy()
valid_df["Class"] = np.asarray(y_val, dtype=np.int64)
all_df = pd.concat([train_df, valid_df], ignore_index=True)
train_positions = list(range(len(train_df)))
valid_positions = list(range(len(train_df), len(all_df)))

to = TabularPandas(
    all_df,
    procs=[],
    cat_names=[],
    cont_names=FEATURES,
    y_names="Class",
    y_block=CategoryBlock(vocab=[0, 1]),
    splits=(train_positions, valid_positions),
)
dls = to.dataloaders(bs=4096, num_workers=0)

n_normal = int((np.asarray(y_train) == 0).sum())
n_fraud = int((np.asarray(y_train) == 1).sum())
class_weights = torch.tensor([1.0, n_normal / n_fraud], dtype=torch.float32)
learn = tabular_learner(
    dls,
    layers=[64, 32],
    config=tabular_config(ps=[0.2, 0.2]),
    loss_func=CrossEntropyLossFlat(weight=class_weights),
)

train_start = time.perf_counter()
lr_suggestion = learn.lr_find(suggest_funcs=(valley,))
lr_max = float(lr_suggestion.valley)
learn.fit_one_cycle(
    30,
    lr_max=lr_max,
    cbs=[EarlyStoppingCallback(monitor="valid_loss", patience=5)],
)
train_seconds = time.perf_counter() - train_start

predict_start = time.perf_counter()
p_val_tensor, _ = learn.get_preds(dl=dls.valid)
test_dl = learn.dls.test_dl(X_test.loc[:, FEATURES].reset_index(drop=True))
p_test_tensor, _ = learn.get_preds(dl=test_dl)
predict_seconds = time.perf_counter() - predict_start
p_val = p_val_tensor[:, 1].numpy()
p_test = p_test_tensor[:, 1].numpy()

(ROOT / "models").mkdir(exist_ok=True)
learn.export(str(ROOT / "models" / "fastai.pkl"))
hardware = (
    "GPU: " + torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU: " + (platform.processor() or platform.machine())
)
fastai_result = evaluate_model(
    "fastai", y_val, p_val, y_test, p_test, row_id_val, row_id_test,
    train_seconds, predict_seconds,
    notes=(
        "fastai TabularPandas with the shared train/validation rows; "
        "weighted CrossEntropyLossFlat; layers [64, 32]; "
        "lr_find valley; up to 30 one-cycle epochs with patience 5. "
        "Training time includes lr_find."
    ),
    hardware=hardware,
    output_dir=ROOT,
)
fastai_result


threshold,precision,recall,f1,pr_auc,roc_auc,rmse,mae,r2,train_seconds,predict_seconds
0.997858,0.883721,0.8,0.839779,0.737738,0.961233,0.086885,0.026905,-3.51678,6.024937,0.142248


In [2]:
def predict_fastai_prob(X):
    dl = learn.dls.test_dl(X.loc[:, FEATURES].reset_index(drop=True))
    probs, _ = learn.get_preds(dl=dl)
    return probs[:, 1].numpy()

fastai_importance = permutation_importance(
    predict_fastai_prob, X_val, y_val, n_repeats=3, random_state=SEED
)
fastai_importance.to_csv(ROOT / "results" / "perm_fastai.csv", index=False)
fastai_importance.head(10)


feature,importance_mean,importance_std,baseline_pr_auc
V14,0.114654,0.005551,0.688709
V12,0.037753,0.003438,0.688709
V10,0.023718,0.003525,0.688709
V5,0.019004,0.001141,0.688709
V4,0.016220,0.002582,0.688709
V17,0.009614,0.003752,0.688709
V16,0.004727,0.000833,0.688709
V11,0.004389,0.000184,0.688709
V8,0.003219,0.000152,0.688709
V15,0.003020,0.001250,0.688709
